# 01 - Dataset audit
Read-only audit of the three datasets. The primary Parquet and the StudentLife archive are audited here; **WESAD is audited locally** with `src/wesad_preprocessing.py --audit-only` (see notebook 03).

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 80)
from src.utils import load_config, reports_dir, outputs_dir
cfg = load_config()          # uses $DEPRESSION_CONFIG if set, else configs/config.yaml
print("config:", cfg["_config_path"])

config: /tmp/config_run.yaml


## Primary dataset (Parquet + TSV)

In [2]:
from src.audit import audit_primary, assemble_inventory
summary = audit_primary(cfg)
pd.Series({k: v for k, v in summary.items() if not isinstance(v, (dict, list))})

n_rows                                 35694
n_feature_columns                        154
n_participants                          4948
labelled_rows                          10866
labelled_participants                   4036
participants_with_>=2_labelled_rows     3107
participants_without_any_label           912
tsv_rows_without_description              84
exact_duplicate_rows                       0
duplicate_feature_vectors                  0
n_corr_pairs_over_threshold                4
columns_over_50pct_missing                10
dtype: int64

In [3]:
print('labels exist only on periods:', summary['label_period_distribution'])
print('PHQ-9 category columns match the standard bands:', summary['category_matches_standard_bands'])
print('TSV <-> parquet name mismatches:', summary['tsv_names_missing_in_parquet'], summary['parquet_columns_missing_in_tsv'])

labels exist only on periods: {np.int64(3): 3244, np.int64(6): 2832, np.int64(9): 2732, np.int64(12): 2058}
PHQ-9 category columns match the standard bands: {'phq9_cat_start': 1.0, 'phq9_cat_end': 1.0}
TSV <-> parquet name mismatches: [] []


In [4]:
rep = reports_dir(cfg)
pd.read_csv(rep / 'invalid_values_report.csv')

,check,n_rows,action
0,weight < min adult weight,20,set weight & bmi to missing
1,bmi > 60 (plausible severe obesity),178,"kept, flagged only"
2,bmi < 14,20,same rows as low weight
3,bmi differs from 703*weight/height^2 by > 1,0,"none found: bmi is derived, redundant"
4,sleep asleep == 0 and in-bed == 0 (weekday),1206,treated as device non-wear -> missing
5,steps_awake_mean < 500,20,"kept, flagged (may be non-wear)"
6,steps_awake_mean > 40000,26,"kept, flagged (extreme but possible)"
7,asleep minutes > in-bed minutes (weekday),0,none found
8,sleep ratio > 1,0,none found
9,negative steps,0,none found


In [5]:
pd.read_csv(rep / 'missing_values_report.csv').head(15)

,column,dtype,n_missing,pct_missing,n_unique
0,steps_mvpa_iqr,float64,30801,86.292,352
1,steps__mvpa__sum__coeff_,float64,27064,75.822,2833
2,steps__mvpa__sum__intercept_,float64,27064,75.822,5533
3,steps__mvpa__sum__score_,float64,27064,75.822,8475
4,phq9_score_start,float64,24828,69.558,28
5,phq9_cat_start,float64,24828,69.558,5
6,phq9_cat_end,float64,24828,69.558,5
7,phq9_score_end,float64,24828,69.558,28
8,sleep_main_start_hour_adj_iqr,float64,20162,56.486,71
9,sleep_ratio_asleep_in_bed_iqr,float64,20162,56.486,15494


In [6]:
pd.read_csv(rep / 'redundant_features.csv')

,feature_a,feature_b,abs_corr
0,sleep__main_efficiency__intercept,sleep_ratio_asleep_in_bed__intercept,0.995903
1,sleep__main_efficiency__intercept_,sleep_ratio_asleep_in_bed__intercept_,0.994174
2,sleep_ratio_asleep_in_bed_weekday_mean,sleep_ratio_asleep_in_bed__intercept,0.982540
3,sleep_ratio_asleep_in_bed_weekday_mean,sleep_ratio_asleep_in_bed_mean_recent,0.982423


In [7]:
d = pd.read_csv(rep / 'feature_data_dictionary.csv')
print(d.role.value_counts()); print(d.tsv_subcategory.value_counts())
d.head()

role
model_candidate                      123
timing_uncertain_sensitivity_set      21
sensitive_fairness_only                6
excluded_outcome_period                3
baseline_input_stored_with_labels      1
Name: count, dtype: int64
tsv_subcategory
Linear regression    84
Static               29
Statistics           27
Dynamic              10
PHQ-9 category        2
PHQ-9 score           2
Name: count, dtype: int64


,feature,tsv_category,tsv_subcategory,tsv_description,tsv_notes,in_parquet,dtype,pct_missing,n_unique,role,has_description,is_constant,raw_or_engineered
0,sex,Demographic,Static,Sex,"0=female, 1=male, 2=other",True,int64,0.0,3,sensitive_fairness_only,True,False,engineered_aggregate_or_static
1,race_white,Demographic,Static,"Race, white",NaN,True,bool,0.0,2,sensitive_fairness_only,True,False,engineered_aggregate_or_static
2,race_black,Demographic,Static,"Race, black",NaN,True,bool,0.0,2,sensitive_fairness_only,True,False,engineered_aggregate_or_static
3,race_hispanic,Demographic,Static,"Race, hispanic",NaN,True,bool,0.0,2,sensitive_fairness_only,True,False,engineered_aggregate_or_static
4,race_asian,Demographic,Static,"Race, asian",NaN,True,bool,0.0,2,sensitive_fairness_only,True,False,engineered_aggregate_or_static


## Archive dataset (identified as StudentLife)

In [8]:
from src.studentlife_audit import audit_studentlife
sl = audit_studentlife(cfg)
print(sl['identified_as']); print(sl['phq9']); print(sl['merge_with_primary'])
pd.read_csv(rep / 'studentlife_inventory.csv')

StudentLife (Dartmouth): per-user u00..u59 phone sensing, EMA, surveys, app usage, calls, SMS
{'responses': 84, 'participants': 46, 'pre_responses': 46, 'post_responses': 38, 'participants_with_pre_and_post': 38, 'incomplete_responses': 0, 'pre_mean': 5.521739130434782, 'post_mean': 6.2631578947368425, 'pre_ge10': 8, 'post_ge10': 7}
NO - different population, sensors, features, label timing and scale of labels


,modality,files,users,uncompressed_MB,extensions
0,EMA/EMA_definition.json,1,0,0.03,json
1,EMA/response,1274,49,2.46,json
2,app_usage,49,49,449.23,csv
3,calendar,28,28,0.19,csv
4,call_log,49,49,7.96,csv
5,dinning,31,31,0.31,txt
6,education,5,0,0.04,"csv,json"
7,sensing/activity,49,49,319.79,csv
8,sensing/audio,49,49,1390.18,csv
9,sensing/bluetooth,49,49,52.91,csv


## Combined inventory

In [9]:
assemble_inventory(cfg)

,dataset,item,kind,rows,columns,notes
0,primary,anon_processed_df_parquet,parquet,35694,154,4948 participants; index = participant_period;...
1,primary,Feature_explanations_-_anonymized_data_-_Sheet...,tsv,154,5,feature dictionary; 84 of 154 rows have no des...
2,studentlife,archive__2_.zip,zip,1984,NaN,3.04 GB uncompressed; 49 users in sensing; PHQ...
3,wesad,local WESAD folder (~2.5 GB compressed),folder of S*/S*.pkl,None,None,NOT accessible in the execution environment - ...
